# LSTM Đa Biến với Entity Embedding

**Target**: `Units Sold`  
**Approach**: Entity Embedding cho 4 static categorical features (`Store ID`, `Product ID`, `Category`, `Region`) + numerical features (bao gồm `Weather Condition`, `Seasonality` dạng số) → LSTM multi-step forecast  
**Horizons**: 7, 14, 28 ngày

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import os
import kagglehub


tf.random.set_seed(42)
np.random.seed(42)

DATA_DIR = kagglehub.dataset_download("atomicd/retail-store-inventory-and-demand-forecasting")
DATA_PATH = os.path.join(DATA_DIR, "sales_data.csv")

#DATA_PATH  = "/kaggle/input/datasets/thaonngyn/retail-store-inventory-and-demand-forecasting/sales_data.csv"
RESULT_DIR = "/kaggle/working/"
TARGET     = 'Units Sold'
TRAIN_END  = '2023-06-30'
VAL_END    = '2023-10-31'
HORIZONS   = [7, 14, 28]
LOOKBACK   = 30
LAG        = 7

2026-03-19 16:34:46.082893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773938086.366934      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773938086.454226      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773938087.210613      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773938087.210656      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773938087.210659      55 computation_placer.cc:177] computation placer alr

## 1. Load & Feature Engineering

In [2]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# ── Static categorical → entity embedding ────────────────────────────────────
CAT_COLS = ['Store ID', 'Product ID', 'Category', 'Region']
cat_vocabs = {}
for col in CAT_COLS:
    uniq = sorted(df_raw[col].unique())
    cat_vocabs[col] = {v: i for i, v in enumerate(uniq)}
    df_raw[col + '_enc'] = df_raw[col].map(cat_vocabs[col])

# ── Time-varying categoricals → ordinal encode vào numerical sequence ────────
WEATHER_MAP = {'Sunny': 0, 'Cloudy': 1, 'Rainy': 2, 'Snowy': 3, 'Windy': 4, 'Stormy': 5}
SEASON_MAP  = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Fall': 3}
df_raw['weather_enc']  = df_raw['Weather Condition'].map(WEATHER_MAP).fillna(0).astype(int)
df_raw['season_enc']   = df_raw['Seasonality'].map(SEASON_MAP).fillna(0).astype(int)

# ── Numerical features ────────────────────────────────────────────────────────
grp = df_raw.groupby(['Store ID', 'Product ID'])[TARGET]
df_raw['lag_7']           = grp.shift(7)
df_raw['lag_14']          = grp.shift(14)
df_raw['lag_28']          = grp.shift(28)
df_raw['rolling_mean_7']  = grp.transform(lambda x: x.shift(1).rolling(7).mean())
df_raw['rolling_mean_14'] = grp.transform(lambda x: x.shift(1).rolling(14).mean())
df_raw['day_of_week']     = df_raw['Date'].dt.dayofweek
df_raw['day_of_month']    = df_raw['Date'].dt.day
df_raw['month']           = df_raw['Date'].dt.month
df_raw['is_weekend']      = (df_raw['day_of_week'] >= 5).astype(int)
df_raw = df_raw.bfill().fillna(0)

NUM_COLS = [
    TARGET,
    'Price', 'Discount', 'Competitor Pricing',
    'Inventory Level', 'Units Ordered',
    'Promotion', 'Epidemic',
    'weather_enc', 'season_enc',          # time-varying — giữ trong sequence
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14',
    'day_of_week', 'day_of_month', 'month', 'is_weekend',
]
TARGET_IDX = NUM_COLS.index(TARGET)
ENC_COLS   = [c + '_enc' for c in CAT_COLS]

series_keys = sorted(df_raw.groupby(['Store ID', 'Product ID']).groups.keys())
print(f'Series: {len(series_keys)} | Num features: {len(NUM_COLS)} | Static cat: {len(CAT_COLS)}')
print('Vocab sizes:', {c: len(v) for c, v in cat_vocabs.items()})

Series: 100 | Num features: 19 | Static cat: 4
Vocab sizes: {'Store ID': 5, 'Product ID': 20, 'Category': 5, 'Region': 4}


## 2. Model với Entity Embedding

In [3]:
def embedding_dim(vocab_size):
    """Rule of thumb: min(50, (vocab_size + 1) // 2)"""
    return min(50, (vocab_size + 1) // 2)

def build_model(lookback, n_num, cat_vocab_sizes, horizon, lstm_units=64, dropout=0.2):
    """
    Inputs:
      - num_input : (batch, lookback, n_num)  — numerical sequence
      - cat_input_i: (batch,)                 — one per categorical (static per series)
    """
    num_input = layers.Input(shape=(lookback, n_num), name='num_input')

    # Entity embeddings (static — same value repeated across timesteps)
    cat_inputs, cat_embeds = [], []
    for i, (col, vocab_size) in enumerate(cat_vocab_sizes.items()):
        inp = layers.Input(shape=(1,), name=f'cat_{i}', dtype='int32')
        emb = layers.Embedding(vocab_size, embedding_dim(vocab_size), name='emb_' + col.replace(' ', '_'))(inp)
        emb = layers.Flatten()(emb)
        cat_inputs.append(inp)
        cat_embeds.append(emb)

    # Tile embeddings across timesteps and concat with numerical
    if cat_embeds:
        cat_concat = layers.Concatenate()(cat_embeds) if len(cat_embeds) > 1 else cat_embeds[0]
        emb_dim    = cat_concat.shape[-1]
        cat_tiled  = layers.RepeatVector(lookback)(cat_concat)  # (batch, lookback, emb_dim)
        x = layers.Concatenate(axis=-1)([num_input, cat_tiled])
    else:
        x = num_input

    x = layers.LSTM(lstm_units, return_sequences=True)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(lstm_units // 2)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(horizon)(x)

    model = models.Model(inputs=[num_input] + cat_inputs, outputs=out)
    model.compile(optimizer='adam', loss='mse')
    return model

# Vocab sizes theo thứ tự CAT_COLS
cat_vocab_sizes = {col: len(cat_vocabs[col]) for col in CAT_COLS}

# Preview model
demo = build_model(LOOKBACK, len(NUM_COLS), cat_vocab_sizes, horizon=7)
demo.summary()

2026-03-19 16:35:21.291184: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cat_0 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_1 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_2 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_3 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Store_ID        │ (None, 1, 3)      │         15 │ cat_0[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Product_ID      │ (None, 1, 10)     │        200 │ cat_1[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Category        │ (None, 1, 3)      │         15 │ cat_2[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Region          │ (None, 1, 2)      │          8 │ cat_3[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 3)         │          0 │ emb_Store_ID[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 10)        │          0 │ emb_Product_ID[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 3)         │          0 │ emb_Category[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 2)         │          0 │ emb_Region[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 18)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ flatten_2[0][0],  │
│                     │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_input           │ (None, 30, 19)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 30, 18)    │          0 │ concatenate[0][0] │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 30, 37)    │          0 │ num_input[0][0],  │
│ (Concatenate)       │                   │            │ repeat_vector[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 30, 64)    │     26,112 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 64)    │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ lstm_1[0][0]    

 Total params: 38,997 (152.33 KB)

 Trainable params: 38,997 (152.33 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
import optuna
import tensorflow as tf
import numpy as np
import random
from tensorflow.keras import callbacks

SUBSET_SERIES = series_keys[:10]   

def objective(trial):

    tf.keras.backend.clear_session()

    np.random.seed(42)
    tf.random.set_seed(42)
    random.seed(42)

    params = {
        "lstm_units": trial.suggest_categorical("lstm_units", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.3),
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "lookback": trial.suggest_categorical("lookback", [14, 30]),
    }

    H_TUNE = [7, 14, 28]   
    horizon_scores = []
    for horizon in H_TUNE:

        
        X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers = \
            build_global_arrays(horizon, params["lookback"])

       
        model = build_model(
            params["lookback"],
            len(NUM_COLS),
            cat_vocab_sizes,
            horizon=horizon,
            lstm_units=params["lstm_units"],
            dropout=params["dropout"]
        )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=params["lr"]),
            loss='mse'
        )

        cb = callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )

        
        model.fit(
            [X_num_tr] + X_cats_tr, y_tr,
            validation_data=([X_num_vl] + X_cats_vl, y_vl),
            epochs=20,
            batch_size=params["batch_size"],
            callbacks=[cb],
            verbose=0
        )

        
        smapes = []

        for store, product in SUBSET_SERIES:
            key = f'{store}_{product}'
            r = rolling_eval(model, scalers[key], store, product, horizon, params["lookback"])
            smapes.append(r['smape'])

        horizon_scores.append(np.mean(smapes))

    return float(np.mean(horizon_scores))

## 3. Build Dataset

In [5]:
def make_sequences(num_arr, cat_row, target_idx, lookback, horizon, stride=7):
    """
    num_arr  : (T, n_num) scaled numerical
    cat_row  : (n_cat,)   encoded categorical values (static for this series)
    Returns  : X_num (N, lookback, n_num), X_cats list of (N,), y (N, horizon)
    """
    X_num, y = [], []
    for i in range(lookback, len(num_arr) - horizon + 1, stride):
        X_num.append(num_arr[i - lookback:i])
        y.append(num_arr[i:i + horizon, target_idx])
    X_num = np.array(X_num, dtype=np.float32)
    y     = np.array(y, dtype=np.float32)
    n     = len(X_num)
    X_cats = [np.full(n, cat_row[j], dtype=np.int32) for j in range(len(cat_row))]
    return X_num, X_cats, y


def build_global_arrays(horizon, lookback):
    X_num_tr, X_num_vl = [], []
    X_cats_tr = [[] for _ in CAT_COLS]
    X_cats_vl = [[] for _ in CAT_COLS]
    y_tr, y_vl = [], []
    scalers = {}

    for store, product in series_keys:
        sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)]
        sdf = sdf.set_index('Date')
        key = f'{store}_{product}'

        num_data = sdf[NUM_COLS].values.astype(np.float32)
        cat_row  = sdf[ENC_COLS].iloc[0].values.astype(np.int32)

        train_num = sdf[:TRAIN_END][NUM_COLS].values.astype(np.float32)
        val_num   = sdf[:VAL_END][NUM_COLS].values.astype(np.float32)

        scaler = StandardScaler().fit(train_num)
        scalers[key] = scaler

        tr_scaled  = scaler.transform(train_num)
        val_scaled = scaler.transform(val_num)

        Xn_tr, Xc_tr, yt = make_sequences(tr_scaled,  cat_row, TARGET_IDX, lookback, horizon)
        Xn_vl, Xc_vl, yv = make_sequences(val_scaled, cat_row, TARGET_IDX, lookback, horizon)

        X_num_tr.append(Xn_tr); X_num_vl.append(Xn_vl)
        y_tr.append(yt);        y_vl.append(yv)
        for j in range(len(CAT_COLS)):
            X_cats_tr[j].append(Xc_tr[j])
            X_cats_vl[j].append(Xc_vl[j])

    X_num_tr = np.concatenate(X_num_tr)
    X_num_vl = np.concatenate(X_num_vl)
    y_tr     = np.concatenate(y_tr)
    y_vl     = np.concatenate(y_vl)
    X_cats_tr = [np.concatenate(x) for x in X_cats_tr]
    X_cats_vl = [np.concatenate(x) for x in X_cats_vl]

    return X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers


print('Dataset builder ready.')

Dataset builder ready.


## 4. Rolling Evaluation

In [6]:
def rolling_eval(model, scaler, store, product, horizon, lookback):
    sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)].set_index('Date')
    cat_row = sdf[ENC_COLS].iloc[0].values.astype(np.int32)

    full_scaled = scaler.transform(sdf[NUM_COLS].values.astype(np.float32))
    eval_start  = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)
    eval_end    = sdf.index.max()

    all_fc, all_ac = [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        t_loc     = sdf.index.get_loc(t)
        win_start = t_loc - lookback
        if win_start < 0:
            t += pd.Timedelta(days=horizon); continue
        actual = sdf[TARGET][t: t + pd.Timedelta(days=horizon - 1)].values
        if len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue

        X_num  = full_scaled[win_start:t_loc][np.newaxis]  # (1, lookback, n_num)
        X_cats = [np.array([[cat_row[j]]], dtype=np.int32).reshape(1,1) for j in range(len(CAT_COLS))]
        # model cat inputs expect shape (batch,) not (batch,1) — squeeze
        X_cats = [c.flatten() for c in X_cats]
        X_cats_in = [c[np.newaxis] for c in X_cats]  # (1,)

        fc_scaled = model.predict([X_num] + X_cats_in, verbose=0)[0]  # (horizon,)
        dummy = np.zeros((horizon, len(NUM_COLS)), dtype=np.float32)
        dummy[:, TARGET_IDX] = fc_scaled
        fc = np.clip(scaler.inverse_transform(dummy)[:, TARGET_IDX], 0, None)

        all_fc.append(fc)
        all_ac.append(actual.astype(np.float32))
        t += pd.Timedelta(days=horizon)

    if not all_fc:
        return {k: np.nan for k in ['smape', 'mase', 'rmse', 'rmsle']}

    fc_arr, ac_arr = np.array(all_fc), np.array(all_ac)
    train_vals = sdf[TARGET][:TRAIN_END].values.astype(np.float32)
    lag   = min(LAG, len(train_vals) - 1)
    denom = np.mean(np.abs(train_vals[lag:] - train_vals[:-lag])) or 1.0

    return {
        'smape': (2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100,
        'mase' : np.mean(np.abs(fc_arr - ac_arr)) / denom,
        'rmse' : float(np.sqrt(np.mean((fc_arr - ac_arr) ** 2))),
        'rmsle': float(np.sqrt(np.mean((np.log1p(np.clip(fc_arr, 0, None)) - np.log1p(np.clip(ac_arr, 0, None))) ** 2))),
    }


print('Evaluation function ready.')

Evaluation function ready.


## 5. Train & Evaluate

In [7]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

BEST = study.best_params

print("Best params:", BEST)
print("Best SMAPE:", study.best_value)

[I 2026-03-19 16:35:21,941] A new study created in memory with name: no-name-78d305b1-e945-443a-8800-ddb4fc79e1dc
[I 2026-03-19 16:38:50,872] Trial 0 finished with value: 37.00615310668945 and parameters: {'lstm_units': 32, 'dropout': 0.15950895732806547, 'lr': 0.0018486771058690208, 'batch_size': 64, 'lookback': 30}. Best is trial 0 with value: 37.00615310668945.
[I 2026-03-19 16:48:23,634] Trial 1 finished with value: 40.85042953491211 and parameters: {'lstm_units': 128, 'dropout': 0.295614534393568, 'lr': 0.0035106888087672373, 'batch_size': 64, 'lookback': 30}. Best is trial 0 with value: 37.00615310668945.
[I 2026-03-19 16:51:55,386] Trial 2 finished with value: 36.67802429199219 and parameters: {'lstm_units': 64, 'dropout': 0.12889217986340967, 'lr': 0.0004989374875584714, 'batch_size': 64, 'lookback': 14}. Best is trial 2 with value: 36.67802429199219.
[I 2026-03-19 16:56:14,239] Trial 3 finished with value: 37.50215530395508 and parameters: {'lstm_units': 128, 'dropout': 0.2570

Best params: {'lstm_units': 64, 'dropout': 0.12889217986340967, 'lr': 0.0004989374875584714, 'batch_size': 64, 'lookback': 14}
Best SMAPE: 36.67802429199219


In [8]:
import os
os.makedirs(RESULT_DIR, exist_ok=True)

results_optuna = []

for h in HORIZONS:
    print(f'\n=== [OPTUNA] Horizon = {h} ===')
    print('  Building arrays...')

    X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers = \
        build_global_arrays(h, BEST["lookback"])

    print(f'  Train: {X_num_tr.shape} | Val: {X_num_vl.shape}')
    print(f'  Params: {BEST}')

    # Build model với BEST params
    model = build_model(
        BEST["lookback"],
        len(NUM_COLS),
        cat_vocab_sizes,
        horizon=h,
        lstm_units=BEST["lstm_units"],
        dropout=BEST["dropout"]
    )

    cb = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BEST["lr"]),
        loss='mse'
    )

    # Train
    model.fit(
        [X_num_tr] + X_cats_tr, y_tr,
        validation_data=([X_num_vl] + X_cats_vl, y_vl),
        epochs=50,
        batch_size=BEST["batch_size"],
        callbacks=[cb],
        verbose=0
    )

    # Rolling evaluation
    print('  Rolling eval on TEST...')

    scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}

    for i, (store, product) in enumerate(series_keys):
        key = f'{store}_{product}'
        r   = rolling_eval(model, scalers[key], store, product, h, BEST["lookback"])

        for k in scores:
            scores[k].append(r[k])

        print(f"    {store} | {product} | "
              f"sMAPE={r['smape']:.2f}% "
              f"MASE={r['mase']:.4f} "
              f"RMSE={r['rmse']:.2f} "
              f"RMSLE={r['rmsle']:.4f}")

    # Summary
    row = {
        'model':         'LSTM-EntityEmbedding-Optuna',
        'dataset':       'retail_inventory_daily',
        'target':        TARGET,
        'horizon':       h,
        'lookback':      BEST["lookback"],
        'mean_smape':    round(float(np.nanmean(scores['smape'])), 4),
        'median_smape':  round(float(np.nanmedian(scores['smape'])), 4),
        'mean_mase':     round(float(np.nanmean(scores['mase'])), 4),
        'median_mase':   round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmse':     round(float(np.nanmean(scores['rmse'])), 4),
        'median_rmse':   round(float(np.nanmedian(scores['rmse'])), 4),
        'mean_rmsle':    round(float(np.nanmean(scores['rmsle'])), 4),
        'median_rmsle':  round(float(np.nanmedian(scores['rmsle'])), 4),
    }

    results_optuna.append(row)

    print(f"  H={h} | "
          f"sMAPE={row['mean_smape']:.2f}% "
          f"MASE={row['mean_mase']:.4f} "
          f"RMSE={row['mean_rmse']:.2f} "
          f"RMSLE={row['mean_rmsle']:.4f}")


out_path = f'{RESULT_DIR}/lstm_entity_emb_optuna_summary.csv'
pd.DataFrame(results_optuna).to_csv(out_path, index=False)

print(f'\nSaved to {out_path}')
pd.DataFrame(results_optuna)


=== [OPTUNA] Horizon = 7 ===
  Building arrays...
  Train: (7600, 14, 19) | Val: (9300, 14, 19)
  Params: {'lstm_units': 64, 'dropout': 0.12889217986340967, 'lr': 0.0004989374875584714, 'batch_size': 64, 'lookback': 14}
  Rolling eval on TEST...
    S001 | P0001 | sMAPE=34.63% MASE=0.8412 RMSE=39.50 RMSLE=0.4714
    S001 | P0002 | sMAPE=33.51% MASE=0.7694 RMSE=39.97 RMSLE=0.4650
    S001 | P0003 | sMAPE=35.04% MASE=0.7333 RMSE=50.29 RMSLE=0.5605
    S001 | P0004 | sMAPE=42.45% MASE=0.8410 RMSE=39.09 RMSLE=0.7069
    S001 | P0005 | sMAPE=37.21% MASE=0.7534 RMSE=47.17 RMSLE=0.4978
    S001 | P0006 | sMAPE=47.36% MASE=0.7510 RMSE=44.36 RMSLE=0.9057
    S001 | P0007 | sMAPE=41.69% MASE=0.8031 RMSE=52.95 RMSLE=0.6219
    S001 | P0008 | sMAPE=34.77% MASE=0.7707 RMSE=32.51 RMSLE=0.4538
    S001 | P0009 | sMAPE=32.91% MASE=0.8940 RMSE=46.70 RMSLE=0.4409
    S001 | P0010 | sMAPE=34.31% MASE=0.7250 RMSE=29.41 RMSLE=0.4384
    S001 | P0011 | sMAPE=41.03% MASE=0.8520 RMSE=34.04 RMSLE=0.5331
    S

,model,dataset,target,horizon,lookback,mean_smape,median_smape,mean_mase,median_mase,mean_rmse,median_rmse,mean_rmsle,median_rmsle
0,LSTM-EntityEmbedding-Optuna,retail_inventory_daily,Units Sold,7,14,39.5378,40.2808,0.8054,0.8095,41.9066,42.3902,0.6194,0.6179
1,LSTM-EntityEmbedding-Optuna,retail_inventory_daily,Units Sold,14,14,39.8728,40.3701,0.8102,0.8012,41.4210,42.0005,0.6331,0.6256
2,LSTM-EntityEmbedding-Optuna,retail_inventory_daily,Units Sold,28,14,39.0764,39.5457,0.7908,0.7903,40.5168,40.6538,0.6283,0.6068
